# 04 — Tree & Ensemble Models

Decision trees and the ensemble methods (bagging, boosting, stacking, voting) built on top of them.

## Decision Trees

In [ ]:
from sklearn.tree import DecisionTreeClassifier, DecisionTreeRegressor, plot_tree

clf = DecisionTreeClassifier(max_depth=5, min_samples_leaf=10, random_state=42)
clf.fit(X_train, y_train)
clf.predict(X_test)
clf.feature_importances_

import matplotlib.pyplot as plt
plt.figure(figsize=(20, 10))
plot_tree(clf, feature_names=feature_names, class_names=class_names, filled=True, max_depth=3)
plt.show()


## Random Forest (bagging of trees)

In [ ]:
from sklearn.ensemble import RandomForestClassifier, RandomForestRegressor

rf = RandomForestClassifier(
    n_estimators=200, max_depth=None, max_features="sqrt",
    n_jobs=-1, random_state=42, oob_score=True
)
rf.fit(X_train, y_train)
rf.oob_score_          # out-of-bag validation score, free estimate without a held-out set
rf.feature_importances_


## Gradient Boosting

In [ ]:
from sklearn.ensemble import GradientBoostingClassifier, HistGradientBoostingClassifier

gb = GradientBoostingClassifier(n_estimators=100, learning_rate=0.1, max_depth=3, random_state=42)
gb.fit(X_train, y_train)

# HistGradientBoosting: much faster on large datasets, native NaN support, LightGBM-like binning
hgb = HistGradientBoostingClassifier(max_iter=200, learning_rate=0.1, early_stopping=True)
hgb.fit(X_train, y_train)


## AdaBoost

In [ ]:
from sklearn.ensemble import AdaBoostClassifier
from sklearn.tree import DecisionTreeClassifier

ada = AdaBoostClassifier(
    estimator=DecisionTreeClassifier(max_depth=1),   # stumps as weak learners
    n_estimators=100, learning_rate=1.0, random_state=42
)
ada.fit(X_train, y_train)


## Voting Ensembles (combine different model types)

In [ ]:
from sklearn.ensemble import VotingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC

voting = VotingClassifier(
    estimators=[
        ("lr", LogisticRegression(max_iter=1000)),
        ("rf", RandomForestClassifier(n_estimators=100)),
        ("svc", SVC(probability=True)),
    ],
    voting="soft",   # average predicted probabilities; "hard" = majority vote on labels
)
voting.fit(X_train, y_train)


## Stacking (meta-model learns to combine base models)

In [ ]:
from sklearn.ensemble import StackingClassifier

stack = StackingClassifier(
    estimators=[
        ("rf", RandomForestClassifier(n_estimators=100)),
        ("gb", GradientBoostingClassifier()),
    ],
    final_estimator=LogisticRegression(),
    cv=5,
)
stack.fit(X_train, y_train)


## Feature Importance Comparison

In [ ]:
importances = pd.Series(rf.feature_importances_, index=feature_names).sort_values(ascending=False)
importances.head(10).plot.barh()
plt.gca().invert_yaxis()
plt.show()

# permutation importance — more reliable than impurity-based importance for correlated features
from sklearn.inspection import permutation_importance
perm = permutation_importance(rf, X_test, y_test, n_repeats=10, random_state=42)
perm.importances_mean


## Common Gotchas
- Impurity-based `feature_importances_` is biased toward high-cardinality features — prefer `permutation_importance` when in doubt.
- Deep unconstrained decision trees overfit badly; always tune `max_depth`/`min_samples_leaf` or use an ensemble.
- `oob_score_` in RandomForest gives a free validation estimate but requires `bootstrap=True` (the default).
- Gradient boosting is sequential (slower to train) and more sensitive to `learning_rate`/`n_estimators` tuning than bagging methods like Random Forest.